#### Analysis 1 

In [1]:
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
fold = 0
exp_path = "./experiments/troubleshooting/20260227_103603"
fold_path = os.path.join(exp_path, "fold_" + str(fold))

In [ ]:
# Confusion matrix
cm = pd.read_csv(os.path.join(fold_path, "confusion_matrix.csv"), index_col=0)

lbls = ["CN", "MCI", "AD"]
cm.index = lbls
cm.columns = lbls

cmap = sn.light_palette('seagreen', as_cmap=True)
sn.heatmap(cm/cm.sum().sum(), annot=True, cmap=cmap)


In [ ]:
# Diagnosis distribution

sn.set_theme(palette='pastel')
samples = pd.read_csv(os.path.join(exp_path, "samples.csv"))
diag_dist = samples.groupby("label")["PET"].nunique()

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    diag_dist,
    labels=["CN", "MCI", "AD"],
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()

In [ ]:
# Train, validation and test splits

sn.set_theme(palette='pastel')

splits = [ 4327, 1050, 586]
lbls = ["Training", "Validation", "Test"]

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    splits,
    labels=lbls,
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()


In [ ]:
# Confidence distributions 

preds = pd.read_csv(os.path.join(fold_path, "predictions.csv"))

lbls = ["CN", "MCI", "AD"]

fig, axes = plt.subplots(3,3, figsize=(15,15))

for i in range(3):
    for j in range(3):

        axes[i,j].set_title(f"true {lbls[i]}, pred { lbls[j] }")
        axes[i,j].hist( 
            preds[ (preds["target"] == i) & (preds["pred"] == j) ]["confidence"], 
            bins=np.linspace(0.4,1,20),
            alpha=0.9, 
            cumulative=False)
        
        axes[i,j].set_ylim([0,50])



In [ ]:
import torch
import json 
import yaml

from pkg.utils.instantiate import instantiate
from pkg.training.criterion import build_criterion

In [ ]:
# Read config 
with open(os.path.join(exp_path, "config.yaml")) as f:
    cfg = yaml.safe_load(f)

print(cfg)

In [ ]:
dm = instantiate(cfg["datamodule"])
dm.setup()
dm.set_fold(fold)

In [ ]:
model = instantiate(cfg["model"])
criterion = build_criterion(cfg["criterion"], train_labels=dm.train_labels)

model.set_criterion(criterion)
best_state = torch.load( os.path.join(fold_path, "model.ckpt"), weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(best_state)

In [ ]:
from tqdm import tqdm

model.eval()

preds = []
targets = []
confs = []

for batch in tqdm(dm.test_dataloader()):

    out = model.test_batch(batch, 0)
    preds.append(out['preds'])
    targets.append(out['targets'])
    confs.append(out['confs'])

# Compute global confusion matrix and classification report
preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
confs = torch.cat(confs).numpy()


### Analysis 2

In [1]:
from pkg.utils.report import experiment_report
from pkg.utils.reproducibility import repo_state
from pkg.utils.instantiate import instantiate
from pkg.data.datasets import ADNIDataset
from pkg.training.criterion import build_criterion

from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Subset, DataLoader
from tqdm import tqdm 

import os
import yaml
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [33]:
#def analyze_predictions(path, fold): 
path = "./experiments/fdg_mri_demog_complete_only/20260319_043211/"
fold = 4

# Read config 
with open(os.path.join(path, "config.yaml")) as f: 
    cfg = yaml.load(f, yaml.FullLoader)

# Read commit
with open(os.path.join(path, "commit.txt")) as f:
    commit = f.read().strip()

# Read patch 
with open(os.path.join(path, "patch.diff"), "rb") as f:
    patch = f.read()

# Read indices 
with open(os.path.join(path, "indices.json")) as f:
    indices = json.loads(f.read())        

# Create dataset with saved samples
ds_cfg = cfg["datamodule"]["args"]["data"]
ds = ADNIDataset(
    data_dir=ds_cfg["data_dir"],
    cached_samples= os.path.join(path, "samples.csv"),
    modalities={"PET-fdg":"", "MRI":""},
)
ds.setup()

# Load model 
with repo_state(commit, patch):
    model = instantiate(cfg["model"])

# Setup model 
model.set_criterion(nn.CrossEntropyLoss(weight=torch.tensor([1,1,1])))

# Load state dict 
with open( os.path.join(path, f"fold_{fold}", "model_stage_0.ckpt"), "rb") as f:
    best_state = torch.load(f, map_location=torch.device("cpu"))
model.load_state_dict(best_state)
model.eval()

Verifying scans available in dir...


PoE(
  (experts): ModuleList(
    (0-1): 2 x Expert(
      (net): Sequential(
        (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        (1): GroupNorm(4, 8, eps=1e-05, affine=True)
        (2): Swish()
        (3): ResidualBlock(
          (residual): Sequential(
            (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (1): GroupNorm(8, 8, eps=1e-05, affine=True)
            (2): Swish()
            (3): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (4): GroupNorm(8, 8, eps=1e-05, affine=True)
            (5): SAM3d(
              (conv): Conv3d(2, 1, kernel_size=(7, 7, 7), stride=(1, 1, 1), padding=(3, 3, 3))
            )
          )
          (skip1): Conv3d(8, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1))
        )
        (4): ResidualBlock(
          (residual): Sequential(
            (0): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))


In [34]:
# Test dataset 
ds_test = Subset(ds, indices["test_idx"])

# Loader 
test_loader = DataLoader(ds_test, batch_size=1, shuffle=False)

In [36]:
# Test dataset 
ds_test = Subset(ds, indices["test_idx"])

# Loader 
test_loader = DataLoader(ds_test, batch_size=1, shuffle=False)

subjects = []
preds = []
targets = []
confs = []

with torch.no_grad():
    for i, batch in tqdm(enumerate(test_loader)):

        batch["mask"][:,0] = 0 # Disable MRI
        out = model.test_batch(batch,i)

        subjects.append(batch["subject"][0])
        preds.append(out["preds"].detach().cpu().view(-1))
        targets.append(out["targets"].detach().cpu().view(-1))
        confs.append(out["confs"].detach().cpu().view(-1))
        
# Compute global confusion matrix and classification report
preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
confs = torch.cat(confs).numpy()

cm = pd.DataFrame(confusion_matrix(targets, preds))
report = pd.DataFrame(classification_report(targets, preds, digits=4, output_dict=True)).T

print(cm)
print(report)

308it [04:13,  1.22it/s]

    0   1   2
0  48  27   2
1  51  70  33
2   3   9  65
              precision    recall  f1-score     support
0              0.470588  0.623377  0.536313   77.000000
1              0.660377  0.454545  0.538462  154.000000
2              0.650000  0.844156  0.734463   77.000000
accuracy       0.594156  0.594156  0.594156    0.594156
macro avg      0.593655  0.640693  0.603079  308.000000
weighted avg   0.610336  0.594156  0.586925  308.000000


In [12]:

true_label = 2
pred_label = 1
lbls = ["CN", "MCI", "AD"]

selected_subj = set({})
converted = set({})

df_dx = pd.read_csv("/project/aereditato/cestari/adni-mri-classification/data/preprocessing_multimodal/csv/adni/DXSUM_02Feb2026.csv")

for i,subj in enumerate(subjects): 
    if (targets[i] == true_label) and (preds[i] == pred_label):
        selected_subj.add(subj)

        if len(df_dx[ (df_dx['PTID'] == subj) & (df_dx["DIAGNOSIS"] == pred_label+1) ]) > 0:
            converted.add(subj)

print(f"{len(converted)}/{len(selected_subj)} = {len(converted) / len(selected_subj):.2%} of { lbls[true_label] } patients incorrectly predicted as { lbls[pred_label]} also have a { lbls[pred_label]} diagnosis at some point")

7/15 = 46.67% of AD patients incorrectly predicted as MCI also have a MCI diagnosis at some point


In [2]:
import os
import json 
from pkg.utils.reproducibility import repo_state

#def analyze_predictions(path, fold): 
path = "./experiments/.saved/pet_fdg_demog_baseline/20260318_103319/"
fold = 4

# Read commit
with open(os.path.join(path, "commit.txt")) as f:
    commit = f.read().strip()

# Read patch 
with open(os.path.join(path, "patch.diff"), "rb") as f:
    patch = f.read()
    
# Load model 
with repo_state(commit, patch):
    with open("./pkg/models/poe.py","r") as f:
        print(f.read())

import torch
import torch.nn as nn
import torch.nn.functional as F
from .base import BaseModel
from .components import ResidualBlock, Swish, SelfAttention3D
from itertools import combinations

# TODO: [OK] log unimodal losses and multimodal losses separately
# TODO: use weights for different losses (unimodal and multimodal), so that no loss dominates overall.
# TODO: [OK] test batch 
# TODO: [OK] confusion matrix based on stratification


class DemographicsExpert(nn.Module):

    def __init__(self, latent_dim, use_swish=True): 
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 16), 
            Swish() if use_swish else nn.ReLU(),
            nn.Linear(16, 16),
            Swish() if use_swish else nn.ReLU()
        )

        self.mlp_mu = nn.Linear(16, latent_dim)
        self.mlp_logvar = nn.Linear(16, latent_dim) 

    def forward(self, x):
        x = self.net(x)
        mu = self.mlp_mu(x)
        logvar = self.mlp_logvar(x)

        return mu

##### A 

In [79]:
import pandas as pd 
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

In [71]:
preds = pd.read_csv("./experiments/mri_pet_weighted_loss_deterministic/20260508_102011/fold_1/predictions.csv")
preds["key"] = preds["key"].astype(str)

In [72]:
mri_p = preds[ (preds["key"].str.endswith("100")) | (preds["key"].str.endswith("010"))]
pet_p = preds[ (preds["key"].str.endswith("001"))]
both_p = preds[ (preds["key"].str.endswith("011")) | (preds["key"].str.endswith("101")) ]

In [86]:
current = both_p

print( classification_report(current["target"], current["pred"]) )
print( confusion_matrix(current["target"],current["pred"]) )
print(balanced_accuracy_score(current["target"], current["pred"]))


              precision    recall  f1-score   support

           0       0.56      0.05      0.10        91
           1       0.51      0.82      0.63       132
           2       0.68      0.72      0.70        75

    accuracy                           0.56       298
   macro avg       0.58      0.53      0.48       298
weighted avg       0.57      0.56      0.49       298

[[  5  81   5]
 [  4 108  20]
 [  0  21  54]]
0.5310422910422911


### Compose unique dataset for all experiments

PET-only = 374

MRI&PET = 2110

MRI-only = 8440

In [1]:
import pandas as pd 

In [2]:
samples = pd.read_csv("./experiments/mri_pet_weighted_loss_deterministic/20260508_102011/samples.csv")
samples

,Unnamed: 0,PET-fdg,MRI,diagnosis,subject_id,age,gender,MMSE,CDR,strat_key,label
0,0,I1592036,I238627,1.0,002_S_0295,2.042304,0.0,0.320237,-1.004694,1011,0
1,1,NaN,I28560,1.0,002_S_0295,1.422185,0.0,0.320237,-1.004694,1100,0
2,2,NaN,I28561,1.0,002_S_0295,1.422185,0.0,0.320237,-1.004694,1100,0
3,3,NaN,I55275,1.0,002_S_0295,1.497642,0.0,0.879900,-1.004694,1100,0
4,4,NaN,I55276,1.0,002_S_0295,1.497642,0.0,0.879900,-1.004694,1100,0
...,...,...,...,...,...,...,...,...,...,...,...
19091,19091,NaN,I11291622,2.0,941_S_7051,-1.071241,0.0,-0.519257,0.216254,2010,1
19092,19092,NaN,I1588331,1.0,941_S_7074,-0.536750,0.0,0.600069,-1.004694,1010,0
19093,19093,NaN,I10283169,1.0,941_S_7074,-0.350696,0.0,0.320237,-1.004694,1010,0
19094,19094,NaN,I11455284,1.0,941_S_7074,-0.075498,0.0,0.320237,-1.004694,1010,0


In [4]:
mri_only = samples[ (samples["MRI"].notna()) & (samples["PET-fdg"].isna()) ]
pet_only = samples[ (samples["MRI"].isna()) & (samples["PET-fdg"].notna()) ]
mri_pet = samples[ (samples["MRI"].notna()) & (samples["PET-fdg"].notna()) ]

In [13]:
mri_only.sample(frac=0.51).reset_index(drop=True)

,Unnamed: 0,PET-fdg,MRI,diagnosis,subject_id,age,gender,MMSE,CDR,strat_key,label
0,13605,NaN,I341919,1.0,100_S_4469,-1.092324,0.0,0.879900,-1.004694,1010,0
1,10788,NaN,I37256,3.0,067_S_1185,-1.722986,0.0,-1.918414,1.437201,3010,2
2,13739,NaN,I141007,2.0,109_S_1183,1.010128,0.0,0.600069,0.216254,2100,1
3,14775,NaN,I90727,2.0,126_S_0709,1.420705,0.0,0.600069,0.216254,2100,1
4,16864,NaN,I99451,1.0,131_S_1301,-0.213467,1.0,0.879900,-1.004694,1010,0
...,...,...,...,...,...,...,...,...,...,...,...
8467,15475,NaN,I390652,3.0,127_S_5095,-1.138930,0.0,-1.638583,1.437201,3010,2
8468,11546,NaN,I257005,2.0,072_S_4063,-2.072902,1.0,0.600069,0.216254,2010,1
8469,5425,NaN,I48100,1.0,023_S_0926,-0.390274,1.0,0.320237,-1.004694,1100,0
8470,8206,NaN,I269243,2.0,035_S_2074,-1.599443,0.0,0.040406,-1.004694,2010,1


In [17]:
# Create new df
df_final = pd.concat([mri_pet, 
           pet_only,
           mri_only.sample(frac=0.52)]).reset_index(drop=True)

In [22]:
df_final.to_csv("./final.csv")

In [21]:
df_final[ (df_final["PET-fdg"].isna()) & (df_final["MRI"].notna())]

,Unnamed: 0,PET-fdg,MRI,diagnosis,subject_id,age,gender,MMSE,CDR,strat_key,label
2484,1715,NaN,I1327191,1.0,007_S_6310,-0.727613,1.0,0.600069,-1.004694,1010,0
2485,14572,NaN,I11298686,2.0,123_S_10796,-0.765712,0.0,0.879900,0.216254,2010,1
2486,831,NaN,I1246020,2.0,003_S_6479,-1.800663,1.0,0.600069,0.216254,2010,1
2487,18453,NaN,I505759,1.0,153_S_4125,0.684995,1.0,0.879900,-1.004694,1010,0
2488,18283,NaN,I310974,2.0,141_S_4053,0.204508,0.0,0.040406,0.216254,2010,1
...,...,...,...,...,...,...,...,...,...,...,...
11117,16042,NaN,I310621,3.0,128_S_4772,0.525203,1.0,-1.918414,0.216254,3010,2
11118,9649,NaN,I330743,2.0,041_S_4513,-1.841351,0.0,0.600069,0.216254,2010,1
11119,9747,NaN,I374482,1.0,041_S_5097,-0.955095,0.0,0.040406,-1.004694,1010,0
11120,11281,NaN,I346482,1.0,068_S_4424,-1.050527,1.0,0.040406,-1.004694,1010,0
